In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display

In [2]:
%load_ext autoreload

In [12]:
#Run this to reload the python file
%autoreload 2
from utils import *

# Preprocess data

In [ ]:
# read the file with the pcp information from GPM and IMN
imn_df = pd.read_csv('./Data/harmonized/unif_IMN.csv')
gpm_df = pd.read_csv('./Data/harmonized/unif_GPM.csv')

In [ ]:
# read the file with the pcp data from GPM and IMN for the Turrialba station
df_tmp_imn = pd.read_csv('./Data/raw/TURR_IMN.csv', sep=';')
df_tmp_gpm = pd.read_csv('/Users/maureenfonseca/Documents/UCR/GPM/Data/GPM_raw/73151_gpm_ST.csv')

In [ ]:
print(len(df_tmp_imn))
print(len(df_tmp_gpm))

In [ ]:
# check the len of the dfs
print(len(imn_df))
print(len(gpm_df))

In [ ]:
# assigning 'date' as index
imn_df = convert_index(imn_df, 'date')
gpm_df = convert_index(gpm_df, 'date')

In [ ]:
df_tmp_gpm

In [ ]:
# assigning 'date' as index
df_tmp_imn = convert_index(df_tmp_imn, 'Date')
df_tmp_gpm = convert_index(df_tmp_gpm, 'date')

In [ ]:
df_tmp_gpm = df_tmp_gpm[:'2021-07-25 23:00:00']

In [ ]:
# rename columns
df_tmp_imn = df_tmp_imn.rename(columns={"pcp": "73151"})
df_tmp_gpm = df_tmp_gpm.rename(columns={"pcp": "73151"})

In [ ]:
# List of columns to select
selected_columns = ['73123', '69679', '71015']

# Select columns from the DataFrame
imn_df = imn_df[selected_columns]
gpm_df = gpm_df[selected_columns]

In [ ]:
# match gpm_df dates with imn_df
imn_df = imn_df['2021-07-21 00:00:00':'2021-07-25 23:00:00']
gpm_df = gpm_df['2021-07-21 00:00:00':'2021-07-25 23:00:00']

In [ ]:
imn_df = pd.concat([imn_df, df_tmp_imn['73151']], axis=1)
gpm_df = pd.concat([gpm_df, df_tmp_gpm['73151']], axis=1)

In [ ]:
imn_df.to_csv('./Data/harmonized/unif_IMN_short_term.csv')
gpm_df.to_csv('./Data/harmonized/unif_GPM_short_term.csv')

In [ ]:
meta = pd.read_csv('./Data/metadata/long_term_v07.csv')

In [ ]:
meta = meta[(meta['Número'] == 73123) | (meta['Número'] == 69679) | (meta['Número'] == 71015)]


In [ ]:
meta = meta.drop(columns=['Unnamed: 0', 'Inicio', 'Fin', 'percentage', 'distance', '1D', '7D', '1M', 'diff_accum', 'percentage_diff'])

In [ ]:
meta = meta.reset_index()

In [ ]:
EMA_to_add = {
    'Número': 73151, 
    'Nombre': 'TURRIALBA CENTRO',
    'Latitud Norte': "09°54'36.4\"", 
    'Longitud Oeste': "83°40'44.4\"", 
    'Altitud (m.s.n.m.)': 657, 
    'lat_x': 9.91, 
    'lon_x': -83.68, 
    'lat_y': 9.95, 
    'lon_y': -83.65, 
    'region_climatica': 'Vertiente del Caribe'
}

In [ ]:
EMA_to_add = pd.DataFrame([EMA_to_add], index=['3'])

In [ ]:
meta = pd.concat([meta, EMA_to_add], axis=0)

In [ ]:
meta = meta.drop(columns=['index'])

In [ ]:
meta = meta.drop(columns=['Unnamed: 0.1'])

In [ ]:
meta

In [ ]:
meta.to_csv('./Data/metadata/meta_medium_term.csv')

# Import Data

In [4]:
imn_df = pd.read_csv('./Data/harmonized/unif_IMN_medium_term.csv')
gpm_df = pd.read_csv('./Data/harmonized/unif_GPM_medium_term.csv')

In [5]:
meta_df = pd.read_csv('./Data/metadata/meta_medium_term.csv')

In [6]:
meta_df = meta_df.sort_values(by='region_climatica', ascending=True)

In [ ]:
#meta_df = meta_df.drop(columns = ['lat_x', 'lon_x'])

In [ ]:
#meta_df.loc[meta_df['Número'] == 71015, 'region_climatica'] = 'Caribe Norte'

In [ ]:
#meta_df.loc[meta_df['Número'] == 73151, 'region_climatica'] = 'Caribe Sur'

In [7]:
# assigning 'date' as index
imn_df = convert_index(imn_df, 'date')
gpm_df = convert_index(gpm_df, 'date')

/opt/homebrew/lib/python3.11/site-packages/pandas/core/indexes/base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)
/opt/homebrew/lib/python3.11/site-packages/pandas/core/indexes/base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


In [8]:
temp_resol = {
    '1H': '1 hora',
    '3H': '3 horas',
    '6H': '6 horas'
}

# Analysis

### Correlations

In [ ]:
corr_temp = []
for window in temp_resol.keys():
    tmp = corr_accum(imn_df, gpm_df, window)
    result_dict = {
        'window': window,
        'correlation': tmp.get('correlation')
    }
    corr_temp.append(result_dict)

In [ ]:
df = pd.DataFrame()
df['1H'] = corr_temp[0]['correlation']
df['3H'] = corr_temp[1]['correlation']
df['6H'] = corr_temp[2]['correlation']

In [ ]:
df = df.reset_index()
df.rename(columns={'index': 'numero'}, inplace=True)

In [ ]:
df['numero'] = df['numero'].astype(float)

In [ ]:
meta_df = pd.merge(meta_df, df, left_on='Número', right_on='numero')
meta_df = meta_df.drop(columns=['numero'])

### Difference between accumulated

In [ ]:
# List to store dictionaries
result_list = []

# Loop through columns and append results to the list
for number in imn_df.columns:
    result_dict = diff_accum(imn_df, gpm_df, number)
    result_list.append(result_dict)

# Convert the list of dictionaries into a DataFrame
accumulated_diff = pd.DataFrame(result_list)

In [ ]:
accumulated_diff['number'] = accumulated_diff['number'].astype(float)

In [ ]:
meta_df = pd.merge(meta_df, accumulated_diff, left_on='Número', right_on='number')
meta_df = meta_df.drop(columns=['Unnamed: 0', 'number'])

In [ ]:
meta_df.drop(columns='Unnamed: 0')

In [ ]:
meta_df.to_csv('./Data/metadata/meta_medium_term.csv')

# Plotting

## Short term

### Time Series

In [9]:
# Create a dropdown widget for selecting the column
column_options = list(imn_df.columns)  # Assuming both dataframes have the same columns
column_dropdown = widgets.Dropdown(
    options=column_options,
    value=column_options[0],
    description='Station Number:',
    disabled=False
)

In [10]:
# Create a dropdown widget for selecting resolution
resolution_options = list(temp_resol.keys()) 
resolution_dropdown = widgets.Dropdown(
    options=resolution_options,
    value=resolution_options[0],
    description='Resolution:',
    disabled=False
)

In [13]:
# Create an interactive plot
interactive_plot = widgets.interactive(
    plot_resol,
    df1=widgets.fixed(imn_df),
    df2=widgets.fixed(gpm_df),
    resolution=resolution_dropdown,
    column=column_dropdown
)

# Display the interactive plot
display(interactive_plot)

interactive(children=(Dropdown(description='Resolution:', options=('1H', '3H', '6H'), value='1H'), Dropdown(de…

### Correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Bar width can be adjusted based on preference
bar_width = 0.2
bar_positions = np.arange(len(meta_df['Número'].unique()))

# Define colors for each bar
color_1d = '#98df8a'  # Light green
color_7d = '#2ca02c'  # Green
color_1m = '#1f771f'  # Dark green

# Plot each column as a separate bar with assigned colors
ax.bar(bar_positions - bar_width, meta_df['1H'], width=bar_width, label='1 hora', color=color_1d)
ax.bar(bar_positions, meta_df['3H'], width=bar_width, label='3 horas', color=color_7d)
ax.bar(bar_positions + bar_width, meta_df['6H'], width=bar_width, label='6 horas', color=color_1m)

# Set labels and title
ax.set_xlabel('Número de estación')
ax.set_ylabel('Correlación')
ax.set_title('Correlaciones según la EMA y el acumulado temporal')
ax.legend()

plt.xticks(bar_positions, meta_df['Número'].unique())
plt.xticks(rotation=45)

plt.tight_layout()

# Show the plot
plt.show()

### Altitude vs Temporal Accumulated Correlation 

In [ ]:
# Create a dropdown widget for selecting the column
column_options = list(temp_resol.keys())
column_dropdown = widgets.Dropdown(
    options=column_options,
    value=column_options[0],
    description='Acumulado:',
    disabled=False
)

In [ ]:
# Create an interactive plot
interactive_plot = widgets.interactive(
    scatter_plot_log_mt,
    df=widgets.fixed(meta_df),
    column=column_dropdown
)

# Display the interactive plot
display(interactive_plot)

### Altitude vs Difference in the accumulated

In [ ]:
plt.figure(figsize=(10, 6))
# Define color map based on unique values in 'region_climatica'
colors = {
    'Valle Central': 'g', 
    'Caribe Norte':'c',
    'Caribe Sur':'teal',
    'Zona Norte':'m' 
    }
    
# Iterate through each region to create scatter plots with appropriate labels
for region, color in colors.items():
    region_data = meta_df[meta_df['region_climatica'] == region]
    plt.scatter((-1)*region_data['percentage_diff'], region_data['Altitud (m.s.n.m.)'], c=color, label=region)

plt.xlabel('[%]')
plt.ylabel('[m.s.n.m] (escala log)')
plt.title(f'Altitud de EMA vs Diferencia Porcentual')
#plt.legend()
plt.yscale('log') 
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# Define color map based on unique values in 'region_climatica'
colors = {
    'Valle Central': 'g', 
    'Caribe Norte':'c',
    'Caribe Sur':'teal',
    'Zona Norte':'m' 
    }
    
# Iterate through each region to create scatter plots with appropriate labels
for region, color in colors.items():
    region_data = meta_df[meta_df['region_climatica'] == region]
    plt.scatter((-1)*region_data['diff_accum'], region_data['Altitud (m.s.n.m.)'], c=color, label=region)

plt.xlabel('[mm]')
plt.ylabel('[m.s.n.m] (escala log)')
plt.title(f'Altitud de EMA vs Diferencia')
plt.legend()
plt.yscale('log') 
plt.show()